# Análisis y Predicción de Ventas de Restaurante

## Proyecto de Data Science para Optimización de Inventario

**Autor:** Cesar Cruz  
**Carrera:** Actuaría con enfoque en Análisis de Datos  
**Fecha:** 2026  
**Tecnologías:** Python · scikit-learn · XGBoost · Pandas · Matplotlib

---
> **Nota:** Este notebook puede ejecutarse con datos reales (`ventas_restaurante.csv`) o con datos simulados automáticamente para demostración.

## Objetivos del Proyecto

- Analizar patrones de ventas diarias y semanales
- Identificar tendencias de demanda para optimizar el inventario
- Comparar tres modelos predictivos: Regresión Lineal, Random Forest y XGBoost
- **Cuantificar la incertidumbre de las predicciones** con intervalos de confianza (enfoque actuarial)
- **Estimar el riesgo de stockout** y calcular el stock de seguridad óptimo
- Generar reportes exportables (JSON, Excel) para toma de decisiones

## 1. Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import warnings
from datetime import datetime
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from scipy import stats
import xgboost as xgb
import holidays
from IPython.display import Markdown, display

warnings.filterwarnings('ignore')
np.random.seed(42)
sns.set_theme(style='whitegrid')

print("✓ Librerías cargadas correctamente")

## 2. Carga y Preparación de Datos

El notebook detecta automáticamente si existe el archivo CSV con datos reales.  
Si no lo encuentra, genera datos simulados para demostración.

In [ ]:
DATA_FILE = 'ventas_restaurante.csv'

def _normalize_col(col: str) -> str:
    return col.strip().lower().replace(' ', '_').replace('(', '').replace(')', '').replace('$', '')

data = None
if os.path.exists(DATA_FILE):
    try:
        with open(DATA_FILE, 'r', encoding='utf-8', errors='replace') as f:
            first_line = f.readline().strip()
            second_line = f.readline().strip()
        first_cols = first_line.count(',') + 1 if first_line else 0
        second_cols = second_line.count(',') + 1 if second_line else 0
    except Exception:
        first_cols = second_cols = 0

    try:
        if second_cols > first_cols and second_cols >= 2:
            data = pd.read_csv(DATA_FILE, header=None,
                               names=['fecha', 'ventas', 'clientes'][:second_cols],
                               parse_dates=['fecha'], dayfirst=True, encoding='utf-8')
        else:
            data = pd.read_csv(DATA_FILE, parse_dates=['fecha'], dayfirst=True, encoding='utf-8')
    except Exception:
        try:
            data = pd.read_csv(DATA_FILE, sep=None, engine='python',
                               parse_dates=['fecha'], dayfirst=True)
        except Exception as e:
            print(f"Error al cargar {DATA_FILE}: {e}")
            data = None

    if data is not None:
        col_map = {}
        normalized = {_normalize_col(c): c for c in data.columns}
        data.columns = [c.strip() for c in data.columns]
        for target, candidates in [('fecha', ['fecha','date','fecha_venta']),
                                   ('ventas', ['ventas','ventas_neta','total','total_amount']),
                                   ('clientes', ['clientes','customers','customer_id'])]:
            if target not in data.columns:
                for cand in candidates:
                    if cand in normalized:
                        col_map[normalized[cand]] = target
                        break
        if col_map:
            data = data.rename(columns=col_map)
        if 'fecha' in data.columns:
            data['fecha'] = pd.to_datetime(data['fecha'], dayfirst=True, errors='coerce')
            data = data.dropna(subset=['fecha'])
        for col in ['ventas', 'clientes']:
            if col in data.columns:
                data[col] = pd.to_numeric(data[col], errors='coerce')

if data is None or 'ventas' not in data.columns:
    print("Archivo no encontrado o inválido → generando datos simulados...")
    fechas = pd.date_range(start="2025-01-01", periods=180)
    tendencia = np.linspace(0, 300, 180)
    estacionalidad = 400 * np.sin(2 * np.pi * fechas.dayofweek / 7)
    ruido = np.random.normal(0, 200, 180)
    ventas_base = 3000 + tendencia + estacionalidad + ruido
    data = pd.DataFrame({
        "fecha": fechas,
        "ventas": np.clip(ventas_base, 1500, 6000).round(2),
        "clientes": np.random.randint(80, 200, 180),
    })
    print(f"✓ Dataset simulado: {len(data)} días")
else:
    print(f"✓ Datos reales cargados: {len(data)} registros")

# Feature engineering
mx_holidays = holidays.Mexico(years=data['fecha'].dt.year.unique())
data['es_festivo']     = data['fecha'].dt.date.isin(mx_holidays)
data['dia_semana']     = data['fecha'].dt.day_name()
data['mes']            = data['fecha'].dt.month
data['es_fin_de_semana'] = data['fecha'].dt.weekday >= 5
data['temp_promedio']  = 25 + 5 * np.sin(2*np.pi*(data['fecha'].dt.dayofyear - 80)/365) + np.random.normal(0, 2, len(data))
data['lluvia']         = np.random.exponential(5, len(data))

# Inventario
INV_FILE = 'inventario_tienda.csv'
if os.path.exists(INV_FILE):
    inv = pd.read_csv(INV_FILE, parse_dates=['fecha'], dayfirst=True)
    inv['fecha'] = pd.to_datetime(inv['fecha'], errors='coerce')
    if 'inventario' not in inv.columns:
        inv.rename(columns={inv.columns[1]: 'inventario'}, inplace=True)
    data = pd.merge(data, inv[['fecha', 'inventario']], on='fecha', how='left')
    print("✓ Inventario real cargado")
else:
    data['inventario'] = np.maximum(0, 1500 - np.cumsum(data['ventas'] / 6) + np.random.normal(0, 30, len(data)))
    print("✓ Inventario simulado generado")

# Outliers (IQR)
Q1, Q3 = data['ventas'].quantile([0.25, 0.75])
IQR = Q3 - Q1
data = data[(data['ventas'] >= Q1 - 1.5*IQR) & (data['ventas'] <= Q3 + 1.5*IQR)].copy()
data['dia_num'] = np.arange(len(data))

print(f"\nDataset final: {data.shape[0]} días | {data['fecha'].min().date()} → {data['fecha'].max().date()}")
print(f"Columnas: {list(data.columns)}")

## 3. Exploración de Datos (EDA)

In [ ]:
data.head(10)

In [ ]:
# Ventas diarias
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(data['fecha'], data['ventas'], marker='o', markersize=3, alpha=0.7, linewidth=1)
axes[0].set_title('Ventas Diarias', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Ventas ($)')
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# Media móvil 7 días
rolling_mean = data['ventas'].rolling(7).mean()
axes[0].plot(data['fecha'], rolling_mean, color='red', linewidth=2, label='Media móvil 7 días', alpha=0.8)
axes[0].legend()

# Distribución
axes[1].hist(data['ventas'], bins=30, color='steelblue', alpha=0.7, edgecolor='white')
axes[1].axvline(data['ventas'].mean(), color='red', linestyle='--', label=f"Media: ${data['ventas'].mean():,.0f}")
axes[1].axvline(data['ventas'].median(), color='orange', linestyle='--', label=f"Mediana: ${data['ventas'].median():,.0f}")
axes[1].set_title('Distribución de Ventas Diarias', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Ventas ($)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Ventas por día de la semana
dias_orden = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dias_es    = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']

promedio_ventas   = data.groupby('dia_semana')['ventas'].mean().reindex(dias_orden)
promedio_clientes = data.groupby('dia_semana')['clientes'].mean().reindex(dias_orden) if 'clientes' in data.columns else None

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(dias_es, promedio_ventas.values, color='steelblue', alpha=0.8)
ax.set_title('Ventas Promedio por Día de la Semana', fontsize=13, fontweight='bold')
ax.set_ylabel('Ventas Promedio ($)')
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, promedio_ventas.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, f'${val:,.0f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

print("\n=== VENTAS PROMEDIO POR DÍA ===")
for d, de, v in zip(dias_orden, dias_es, promedio_ventas.values):
    print(f"  {de:<12}: ${v:,.0f}")

In [ ]:
# Heatmap de correlación
plt.figure(figsize=(9, 6))
cols_corr = ['ventas', 'clientes', 'temp_promedio', 'lluvia', 'es_festivo', 'es_fin_de_semana']
cols_corr = [c for c in cols_corr if c in data.columns]
corr_matrix = data[cols_corr].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', center=0,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Mapa de Correlación entre Variables', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Modelado Predictivo

In [ ]:
FEATURES = ['dia_num', 'clientes', 'temp_promedio', 'lluvia', 'es_festivo', 'mes', 'es_fin_de_semana']
FEATURES  = [f for f in FEATURES if f in data.columns]

X = data[FEATURES]
Y = data['ventas']

modelos = {
    'Regresión Lineal': LinearRegression(),
    'Random Forest':    RandomForestRegressor(n_estimators=200, random_state=42),
    'XGBoost':          xgb.XGBRegressor(n_estimators=200, random_state=42, verbosity=0)
}

tscv = TimeSeriesSplit(n_splits=5)
resultados = {}

for nombre, modelo in modelos.items():
    cv_mae = -cross_val_score(modelo, X, Y, cv=tscv, scoring='neg_mean_absolute_error').mean()
    cv_r2  = cross_val_score(modelo, X, Y, cv=tscv, scoring='r2').mean()
    modelo.fit(X, Y)
    pred   = modelo.predict(X)
    mae    = mean_absolute_error(Y, pred)
    rmse   = np.sqrt(mean_squared_error(Y, pred))
    r2     = modelo.score(X, Y)
    resultados[nombre] = {'MAE': mae, 'RMSE': rmse, 'R²': r2,
                          'MAE_CV': cv_mae, 'R²_CV': cv_r2, 'pred': pred}

print("=== COMPARACIÓN DE MODELOS ===")
print(f"{'Modelo':<20} {'MAE':>8} {'RMSE':>8} {'R²':>7} {'MAE-CV':>9} {'R²-CV':>8}")
print("-"*65)
for n, m in resultados.items():
    print(f"{n:<20} {m['MAE']:>8.1f} {m['RMSE']:>8.1f} {m['R²']:>7.4f} {m['MAE_CV']:>9.1f} {m['R²_CV']:>8.4f}")

# Variables de referencia para celdas siguientes
rf_model       = modelos['Random Forest']
predicciones   = resultados['Regresión Lineal']['pred']
features       = FEATURES
feature_importance = rf_model.feature_importances_
r2_score       = modelos['Regresión Lineal'].score(X, Y)
pendiente_modelo = modelos['Regresión Lineal'].coef_[0]
tendencia      = 'crecimiento' if pendiente_modelo > 0 else 'decrecimiento'

In [ ]:
# Residuos de los 3 modelos
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Análisis de Residuos por Modelo', fontsize=14, fontweight='bold')

for i, (nombre, mod_res) in enumerate(resultados.items()):
    pred    = mod_res['pred']
    residuos = Y - pred
    axes[i].scatter(pred, residuos, alpha=0.5, s=20)
    axes[i].axhline(0, color='red', linestyle='--', linewidth=1)
    axes[i].set_title(nombre, fontsize=11)
    axes[i].set_xlabel('Predicción')
    axes[i].set_ylabel('Residuo')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Importancia de características (Random Forest)
sorted_idx   = np.argsort(feature_importance)
sorted_feats = [features[i] for i in sorted_idx]
sorted_imp   = feature_importance[sorted_idx]

plt.figure(figsize=(9, 5))
bars = plt.barh(sorted_feats, sorted_imp, color='steelblue', alpha=0.8)
plt.xlabel('Importancia (Gini)')
plt.title('Importancia de Variables — Random Forest', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
for bar, val in zip(bars, sorted_imp):
    plt.text(val + 0.001, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Ventas reales vs predicción (Regresión Lineal — tendencia)
plt.figure(figsize=(14, 6))
plt.plot(data['fecha'], data['ventas'], 'o-', alpha=0.5, markersize=3, label='Ventas reales', color='steelblue')
plt.plot(data['fecha'], predicciones, '-', linewidth=2.5, label='Tendencia (Reg. Lineal)', color='red')
plt.title('Ventas Reales vs Tendencia Predicha', fontsize=14, fontweight='bold')
plt.xlabel('Fecha')
plt.ylabel('Ventas ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Modelo de Inventario

In [ ]:
if 'inventario' not in data.columns:
    raise ValueError("La columna 'inventario' no está disponible.")

inv_data = data[['fecha','ventas','clientes','temp_promedio','lluvia',
                 'es_festivo','mes','es_fin_de_semana','inventario']].copy()
inv_data['inventario_next'] = inv_data['inventario'].shift(-1)
inv_data = inv_data.dropna(subset=['inventario_next'])

features_inv = ['ventas','clientes','temp_promedio','lluvia',
                'es_festivo','mes','es_fin_de_semana','inventario']
features_inv = [f for f in features_inv if f in inv_data.columns]

X_inv = inv_data[features_inv]
y_inv = inv_data['inventario_next']

split_index   = int(len(X_inv) * 0.8)
X_inv_train   = X_inv.iloc[:split_index]
X_inv_test    = X_inv.iloc[split_index:]
y_inv_train   = y_inv.iloc[:split_index]
y_inv_test    = y_inv.iloc[split_index:]

model_inv = RandomForestRegressor(n_estimators=200, random_state=42)
model_inv.fit(X_inv_train, y_inv_train)

pred_inv_test = model_inv.predict(X_inv_test)
mae_inv = mean_absolute_error(y_inv_test, pred_inv_test)
r2_inv  = model_inv.score(X_inv_test, y_inv_test)

safety_stock     = 300
today_inv        = X_inv.iloc[-1:]
next_inv_pred    = model_inv.predict(today_inv)[0]
pedido_recomendado = max(0, safety_stock - next_inv_pred)

print("=== MODELO DE INVENTARIO ===")
print(f"MAE  (test): {mae_inv:.2f}")
print(f"R²   (test): {r2_inv:.4f}")
print(f"Inventario estimado mañana: {next_inv_pred:.1f}")
print(f"Pedido sugerido (safety={safety_stock}): {pedido_recomendado:.1f}")

plt.figure(figsize=(12, 5))
plt.plot(inv_data['fecha'].iloc[split_index:], y_inv_test, 'o-', label='Real', markersize=4)
plt.plot(inv_data['fecha'].iloc[split_index:], pred_inv_test, 'x--', label='Predicho', alpha=0.8)
plt.title('Inventario Real vs Predicho (conjunto de prueba)', fontsize=13, fontweight='bold')
plt.xlabel('Fecha')
plt.ylabel('Inventario')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Análisis Actuarial de Riesgo

Esta sección aplica técnicas actuariales para **cuantificar la incertidumbre** en las predicciones y calcular el **riesgo de stockout** bajo distintos niveles de confianza.

- **Intervalos de predicción** con percentiles del Random Forest
- **Value at Risk (VaR)** de ventas al 5%
- **Probabilidad de stockout** bajo demanda estocástica
- **Stock de seguridad óptimo** por nivel de servicio

In [ ]:
# ── INTERVALOS DE PREDICCIÓN (Random Forest por árbol) ──────────────────────
# Cada árbol del bosque genera una predicción independiente → distribución empírica
tree_preds = np.array([tree.predict(X) for tree in rf_model.estimators_])  # shape: (n_trees, n_obs)

p10 = np.percentile(tree_preds, 10, axis=0)
p50 = np.percentile(tree_preds, 50, axis=0)  # mediana
p90 = np.percentile(tree_preds, 90, axis=0)

plt.figure(figsize=(14, 6))
plt.plot(data['fecha'], data['ventas'], 'o', markersize=3, alpha=0.5, color='steelblue', label='Ventas reales')
plt.plot(data['fecha'], p50, '-', linewidth=2, color='darkred', label='Predicción mediana (RF)')
plt.fill_between(data['fecha'], p10, p90, alpha=0.2, color='red', label='Intervalo 80% confianza')
plt.title('Predicciones con Intervalos de Confianza — Random Forest', fontsize=13, fontweight='bold')
plt.xlabel('Fecha')
plt.ylabel('Ventas ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

cobertura = np.mean((data['ventas'].values >= p10) & (data['ventas'].values <= p90))
amplitud  = (p90 - p10).mean()
print(f"Cobertura real del intervalo 80%: {cobertura:.1%}  (esperado ≈ 80%)")
print(f"Amplitud promedio del intervalo:  ${amplitud:,.0f}")


In [ ]:
# ── VALUE AT RISK (VaR) DE VENTAS ────────────────────────────────────────────
# VaR al 5%: el nivel de ventas que solo se supera el 5% peor de los días
VaR_5  = data['ventas'].quantile(0.05)
VaR_1  = data['ventas'].quantile(0.01)
CVaR_5 = data['ventas'][data['ventas'] <= VaR_5].mean()  # Expected Shortfall

# Test de normalidad (Shapiro-Wilk)
stat, p_valor = stats.shapiro(data['ventas'])
distribucion  = "normal" if p_valor > 0.05 else "no normal (cola pesada)"

# Ajuste a distribución normal y log-normal
mu_norm, sigma_norm = data['ventas'].mean(), data['ventas'].std()
VaR_norm = stats.norm.ppf(0.05, loc=mu_norm, scale=sigma_norm)
VaR_logn = np.exp(stats.norm.ppf(0.05, loc=np.log(data['ventas']).mean(),
                                  scale=np.log(data['ventas']).std()))

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(data['ventas'], bins=35, density=True, alpha=0.6, color='steelblue', label='Distribución empírica')
x_range = np.linspace(data['ventas'].min(), data['ventas'].max(), 300)
ax.plot(x_range, stats.norm.pdf(x_range, mu_norm, sigma_norm), 'r-', linewidth=2, label='Ajuste Normal')
ax.axvline(VaR_5, color='orange', linestyle='--', linewidth=2, label=f'VaR 5% = ${VaR_5:,.0f}')
ax.axvline(CVaR_5, color='red', linestyle='--', linewidth=2, label=f'CVaR (ES) 5% = ${CVaR_5:,.0f}')
ax.set_title('Distribución de Ventas · Value at Risk Actuarial', fontsize=13, fontweight='bold')
ax.set_xlabel('Ventas ($)')
ax.set_ylabel('Densidad')
ax.legend()
plt.tight_layout()
plt.show()

print("=== VALUE AT RISK DE VENTAS ===")
print(f"Distribución: {distribucion}  (Shapiro-Wilk p={p_valor:.4f})")
print(f"VaR  5% (empírico):  ${VaR_5:,.0f}  — el 5% de días peores bajan de este nivel")
print(f"VaR  1% (empírico):  ${VaR_1:,.0f}")
print(f"CVaR 5% (ES):        ${CVaR_5:,.0f}  — venta promedio en el 5% peor")
print(f"VaR  5% (Normal):    ${VaR_norm:,.0f}")
print(f"VaR  5% (Log-Normal): ${VaR_logn:,.0f}")


In [ ]:
# ── RIESGO DE STOCKOUT Y STOCK DE SEGURIDAD ÓPTIMO ──────────────────────────
# Modelo: demanda diaria ~ distribución empírica de ventas
# Stock de seguridad = z * sigma_demanda * sqrt(lead_time)

lead_time   = 2   # días de reposición
sigma_d     = data['ventas'].std()
mu_d        = data['ventas'].mean()

niveles_servicio = [0.90, 0.95, 0.99]
z_valores        = [stats.norm.ppf(ns) for ns in niveles_servicio]

print("=== STOCK DE SEGURIDAD ÓPTIMO ===")
print(f"Demanda media diaria: ${mu_d:,.0f}  |  Desv. estándar: ${sigma_d:,.0f}")
print(f"Lead time asumido:    {lead_time} días")
print()
print(f"{'Nivel de servicio':>20} {'z':>7} {'Safety stock':>15} {'Punto de reorden':>18}")
print("-"*65)
for ns, z in zip(niveles_servicio, z_valores):
    ss      = z * sigma_d * np.sqrt(lead_time)
    reorder = mu_d * lead_time + ss
    print(f"{ns:>20.0%} {z:>7.2f} {ss:>15,.0f} {reorder:>18,.0f}")

# Simulación de probabilidad de stockout (Monte Carlo)
N_SIM     = 10_000
inv_ini   = data['inventario'].mean()

def sim_stockout(inventario_inicial, safety_stock_val, n_dias=7):
    stockouts = 0
    for _ in range(N_SIM):
        inv = inventario_inicial + safety_stock_val
        for d in range(n_dias):
            demanda = np.random.choice(data['ventas'].values)
            inv    -= demanda
            if inv < 0:
                stockouts += 1
                break
    return stockouts / N_SIM

print("\n=== PROBABILIDAD DE STOCKOUT (Monte Carlo — 10,000 simulaciones, 7 días) ===")
for ns, z in zip(niveles_servicio, z_valores):
    ss   = z * sigma_d * np.sqrt(lead_time)
    prob = sim_stockout(inv_ini, ss)
    print(f"  Safety stock ${ss:,.0f} (NS={ns:.0%}): P(stockout) = {prob:.2%}")


## 7. Métricas Clave del Negocio

In [ ]:
ventas_totales          = data['ventas'].sum()
ventas_promedio_diarias = data['ventas'].mean()
ventas_maxima           = data['ventas'].max()
ventas_minima           = data['ventas'].min()
dias_analizados         = len(data)

ventas_por_dia      = data.groupby('dia_semana')['ventas'].agg(['mean','max','min','count']).round(2)
dia_mas_rentable    = ventas_por_dia['mean'].idxmax()
dia_menos_rentable  = ventas_por_dia['mean'].idxmin()

dias_orden = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
finde      = ventas_por_dia.reindex([d for d in dias_orden if d in ventas_por_dia.index])
entre      = finde.drop([d for d in ['Saturday','Sunday'] if d in finde.index])
diferencia_fin_semana = (
    finde.loc[['Saturday','Sunday'],'mean'].mean() - entre['mean'].mean()
    if all(d in finde.index for d in ['Saturday','Sunday']) else 0
)

clientes_totales = clientes_promedio_diarios = ticket_promedio = "N/D"
if 'clientes' in data.columns:
    clientes_totales          = data['clientes'].sum()
    clientes_promedio_diarios = data['clientes'].mean()
    ticket_promedio           = (data['ventas'] / data['clientes'].replace(0, np.nan)).mean()

print("=== MÉTRICAS CLAVE ===")
print(f"Período:              {dias_analizados} días")
print(f"Ventas totales:       ${ventas_totales:>12,.0f}")
print(f"Promedio diario:      ${ventas_promedio_diarias:>12,.0f}")
print(f"Máxima / Mínima:      ${ventas_maxima:,.0f} / ${ventas_minima:,.0f}")
print(f"Día más rentable:     {dia_mas_rentable} (${ventas_por_dia.loc[dia_mas_rentable,'mean']:,.0f})")
print(f"Día menos rentable:   {dia_menos_rentable} (${ventas_por_dia.loc[dia_menos_rentable,'mean']:,.0f})")
print(f"Prima fin de semana:  ${diferencia_fin_semana:,.0f}")
if isinstance(ticket_promedio, float):
    print(f"Ticket promedio:      ${ticket_promedio:,.2f}")

## 8. Conclusiones y Recomendaciones

In [ ]:
cv_ventas          = data['ventas'].std() / data['ventas'].mean()
riesgo_modelo      = "bajo" if r2_score >= 0.30 else "moderado" if r2_score >= 0.15 else "alto"
riesgo_volatilidad = "alto" if cv_ventas > 0.35 else "moderado" if cv_ventas > 0.20 else "bajo"

VaR_5_val  = data['ventas'].quantile(0.05)
CVaR_5_val = data['ventas'][data['ventas'] <= VaR_5_val].mean()
ss_95      = stats.norm.ppf(0.95) * data['ventas'].std() * np.sqrt(2)

md = f"""
### Resultados del análisis

| Métrica | Valor |
|---|---|
| Período analizado | {dias_analizados} días |
| Día más rentable | {dia_mas_rentable} (${ventas_por_dia.loc[dia_mas_rentable,'mean']:,.0f} promedio) |
| Día menos rentable | {dia_menos_rentable} |
| Prima fin de semana | ${diferencia_fin_semana:,.0f} |
| R² tendencia (Reg. Lineal) | {r2_score:.1%} |
| VaR 5% ventas | ${VaR_5_val:,.0f} |
| CVaR 5% (Expected Shortfall) | ${CVaR_5_val:,.0f} |
| Stock seguridad óptimo (95%) | {ss_95:,.0f} unidades |

### Recomendaciones estratégicas

- **Inventario:** mantener stock de seguridad ≥ {ss_95:,.0f} unidades para nivel de servicio 95%.
- **Días de mayor demanda:** reforzar abastecimiento previo a {dia_mas_rentable.lower()}s.
- **Riesgo de ingresos:** en el 5% peor de los días las ventas no superan ${VaR_5_val:,.0f}; planificar caja operativa en consecuencia.
- **Próximos pasos:** incorporar modelos de series de tiempo (Prophet/ARIMA) y datos externos (clima real, eventos locales).
"""
display(Markdown(md))

## 9. Exportación de Reportes

In [ ]:
try:
    import openpyxl
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'openpyxl', '-q'])
    import openpyxl

os.makedirs('reportes', exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

# JSON
metricas_dict = {
    "fecha_generacion": datetime.now().isoformat(),
    "periodo": {"dias": dias_analizados,
                "inicio": str(data['fecha'].min().date()),
                "fin":    str(data['fecha'].max().date())},
    "ventas": {"total": round(float(ventas_totales), 2),
               "promedio_diario": round(float(ventas_promedio_diarias), 2),
               "maxima": round(float(ventas_maxima), 2),
               "minima": round(float(ventas_minima), 2)},
    "modelo": {"r2": round(float(r2_score), 4),
               "tendencia": tendencia,
               "pendiente_diaria": round(float(pendiente_modelo), 2)},
    "riesgo_actuarial": {
        "VaR_5pct": round(float(VaR_5_val), 2),
        "CVaR_5pct": round(float(CVaR_5_val), 2),
        "safety_stock_95pct": round(float(ss_95), 0)
    }
}
json_path = f"reportes/reporte_{ts}.json"
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(metricas_dict, f, indent=2, ensure_ascii=False)

# Excel
excel_path = f"reportes/datos_{ts}.xlsx"
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    data.to_excel(writer, sheet_name='Datos', index=False)
    ventas_por_dia.to_excel(writer, sheet_name='Por_Dia')
    pd.DataFrame({
        'fecha': data['fecha'],
        'ventas_reales': data['ventas'],
        'pred_lineal': predicciones,
        'pred_rf_p50': p50,
        'pred_rf_p10': p10,
        'pred_rf_p90': p90,
    }).to_excel(writer, sheet_name='Predicciones', index=False)

# CSV para GitHub (muestra anonimizada)
csv_path = "ventas_muestra.csv"
data[['fecha','ventas','clientes','dia_semana','es_festivo','es_fin_de_semana']].to_csv(
    csv_path, index=False, date_format='%Y-%m-%d')

print(f"✓ JSON:  {json_path}")
print(f"✓ Excel: {excel_path}")
print(f"✓ CSV muestra: {csv_path}")

## 10. Reporte para Power BI

Genera un CSV ejecutivo con todas las métricas en una sola tabla plana —  
listo para conectar directamente desde Power BI con **Obtener datos → Texto/CSV**.

In [ ]:
import os
from datetime import datetime

os.makedirs('reportes', exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

# ── 1. TABLA PRINCIPAL: una fila por día ─────────────────────────────────────
# Incluye ventas reales, predicciones, inventario y bandas de incertidumbre
pbi = data[['fecha', 'ventas', 'clientes', 'dia_semana', 'mes',
            'es_festivo', 'es_fin_de_semana', 'inventario']].copy()

# Predicciones de los 3 modelos
pbi['pred_lineal']   = resultados['Regresión Lineal']['pred'].round(2)
pbi['pred_rf_p50']   = p50.round(2)
pbi['pred_rf_p10']   = p10.round(2)   # límite inferior 80%
pbi['pred_rf_p90']   = p90.round(2)   # límite superior 80%
pbi['pred_xgboost']  = resultados['XGBoost']['pred'].round(2)

# Error de predicción (Random Forest)
pbi['error_pred']    = (pbi['ventas'] - pbi['pred_rf_p50']).round(2)
pbi['error_pct']     = ((pbi['error_pred'] / pbi['ventas']) * 100).round(2)

# Ticket promedio diario
if 'clientes' in data.columns:
    pbi['ticket_promedio'] = (pbi['ventas'] / pbi['clientes'].replace(0, None)).round(2)

# Columna de semana ISO (útil para agrupar en PBI)
pbi['semana_iso']    = pbi['fecha'].dt.isocalendar().week.astype(int)
pbi['anio']          = pbi['fecha'].dt.year

# Inventario: pedido recomendado por día
ss_95 = round(stats.norm.ppf(0.95) * data['ventas'].std() * (2 ** 0.5), 0)
pbi['pedido_recomendado'] = (pbi['inventario'].apply(
    lambda inv: max(0, ss_95 - inv))).round(0)

# ── 2. TABLA KPIs GLOBALES: una fila por métrica ─────────────────────────────
VaR_5_val  = data['ventas'].quantile(0.05)
CVaR_5_val = data['ventas'][data['ventas'] <= VaR_5_val].mean()

kpis = {
    'ventas_totales':          round(float(ventas_totales), 2),
    'ventas_promedio_diario':  round(float(ventas_promedio_diarias), 2),
    'ventas_maxima':           round(float(ventas_maxima), 2),
    'ventas_minima':           round(float(ventas_minima), 2),
    'clientes_promedio_diario': round(float(clientes_promedio_diarios), 2) if isinstance(clientes_promedio_diarios, float) else None,
    'ticket_promedio':         round(float(ticket_promedio), 2) if isinstance(ticket_promedio, float) else None,
    'dias_analizados':         int(dias_analizados),
    'dia_mas_rentable':        dia_mas_rentable,
    'dia_menos_rentable':      dia_menos_rentable,
    'prima_fin_de_semana':     round(float(diferencia_fin_semana), 2),
    'r2_random_forest_cv':     round(float(resultados['Random Forest']['R²_CV']), 4),
    'mae_random_forest_cv':    round(float(resultados['Random Forest']['MAE_CV']), 2),
    'VaR_5pct':                round(float(VaR_5_val), 2),
    'CVaR_5pct':               round(float(CVaR_5_val), 2),
    'safety_stock_95pct':      float(ss_95),
    'inventario_estimado_manana': round(float(next_inv_pred), 1),
    'pedido_sugerido_manana':  round(float(pedido_recomendado), 1),
}

df_kpis = pd.DataFrame([
    {'metrica': k, 'valor': v} for k, v in kpis.items()
])

# ── 3. EXPORTAR ──────────────────────────────────────────────────────────────
path_main = f'reportes/powerbi_ventas_{ts}.csv'
path_kpis = f'reportes/powerbi_kpis_{ts}.csv'

pbi.to_csv(path_main, index=False, date_format='%Y-%m-%d', encoding='utf-8-sig')
df_kpis.to_csv(path_kpis, index=False, encoding='utf-8-sig')

print("✓ Archivos Power BI generados:")
print(f"   {path_main}  →  tabla principal ({len(pbi)} filas, {len(pbi.columns)} columnas)")
print(f"   {path_kpis}  →  KPIs globales  ({len(df_kpis)} métricas)")
print()
print("── Columnas tabla principal ──────────────────────────────")
for col in pbi.columns:
    print(f"   {col}")
print()
print("── Cómo conectar en Power BI ─────────────────────────────")
print("   Inicio → Obtener datos → Texto/CSV → selecciona el archivo")
print("   Delimitador: Coma | Origen: UTF-8")


## 11. Guía de Uso con Datos Reales

### Formato requerido — `ventas_restaurante.csv`

| Columna | Tipo | Descripción |
|---|---|---|
| `fecha` | `YYYY-MM-DD` | Fecha de las ventas |
| `ventas` | numérico | Monto total del día |
| `clientes` | numérico (opcional) | Clientes atendidos |

### Formato opcional — `inventario_tienda.csv`

| Columna | Tipo | Descripción |
|---|---|---|
| `fecha` | `YYYY-MM-DD` | Fecha |
| `inventario` | numérico | Nivel de inventario al cierre |

### Ejemplo rápido

```
fecha,ventas,clientes
2025-01-01,3450,120
2025-01-02,2980,98
2025-01-03,4100,145
```

## 10. Exportación para Power BI

Genera 4 CSVs listos para importar directamente en Power BI Desktop:

| Archivo | Contenido |
|---|---|
| `pbi_kpis_*.csv` | Tarjetas ejecutivas (ventas, ticket, clientes) |
| `pbi_predicciones_*.csv` | Serie temporal: reales vs predicción con intervalos |
| `pbi_riesgo_*.csv` | VaR, CVaR y stock de seguridad por nivel de servicio |
| `pbi_inventario_*.csv` | Inventario diario, alertas de stockout y pedido recomendado |

> Todos los archivos usan **encoding UTF-8 BOM** (`utf-8-sig`) para que Power BI lea correctamente los acentos en español.

In [ ]:
# ═══════════════════════════════════════════════════════
# EXPORTACIÓN PARA POWER BI
# ═══════════════════════════════════════════════════════
os.makedirs('reportes', exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

# ── 1. KPIs principales ──────────────────────────────────────────────────────
kpis_pbi = pd.DataFrame([{
    'kpi':        'Ventas totales',
    'valor':      round(float(ventas_totales), 2),
    'unidad':     'MXN',
    'descripcion': f'Suma de ventas en {dias_analizados} días'
},{
    'kpi':        'Venta promedio diaria',
    'valor':      round(float(ventas_promedio_diarias), 2),
    'unidad':     'MXN',
    'descripcion': 'Promedio diario del período'
},{
    'kpi':        'Ticket promedio',
    'valor':      round(float(ticket_promedio), 2) if isinstance(ticket_promedio, float) else 0,
    'unidad':     'MXN/cliente',
    'descripcion': 'Venta promedio por cliente'
},{
    'kpi':        'Clientes promedio diario',
    'valor':      round(float(data['clientes'].mean()), 1) if 'clientes' in data.columns else 0,
    'unidad':     'clientes',
    'descripcion': 'Promedio de clientes por día'
},{
    'kpi':        'Días analizados',
    'valor':      float(dias_analizados),
    'unidad':     'días',
    'descripcion': f"{data['fecha'].min().date()} → {data['fecha'].max().date()}"
}])

# ── 2. Predicciones vs reales ────────────────────────────────────────────────
predicciones_pbi = pd.DataFrame({
    'fecha':             data['fecha'].dt.strftime('%Y-%m-%d'),
    'ventas_reales':     data['ventas'].round(2),
    'prediccion_rf':     p50.round(2),
    'intervalo_bajo':    p10.round(2),
    'intervalo_alto':    p90.round(2),
    'error_absoluto':    (data['ventas'] - p50).abs().round(2),
    'dentro_intervalo':  ((data['ventas'] >= p10) & (data['ventas'] <= p90)).astype(int),
    'dia_semana':        data['dia_semana'],
    'es_fin_de_semana':  data['es_fin_de_semana'].astype(int),
    'es_festivo':        data['es_festivo'].astype(int),
})

# ── 3. Riesgo actuarial ──────────────────────────────────────────────────────
riesgo_pbi = pd.DataFrame([{
    'metrica':     'VaR 5%',
    'valor':       round(float(VaR_5_val), 2),
    'descripcion': 'Venta mínima esperada el 5% de los días peores',
    'unidad':      'MXN'
},{
    'metrica':     'CVaR 5% (Expected Shortfall)',
    'valor':       round(float(CVaR_5_val), 2),
    'descripcion': 'Venta promedio en el 5% de días peores',
    'unidad':      'MXN'
},{
    'metrica':     'Stock de seguridad 90%',
    'valor':       round(float(ss_95 * 0.85), 0),
    'descripcion': 'Cubre demanda el 90% del tiempo',
    'unidad':      'unidades'
},{
    'metrica':     'Stock de seguridad 95%',
    'valor':       round(float(ss_95), 0),
    'descripcion': 'Cubre demanda el 95% del tiempo',
    'unidad':      'unidades'
},{
    'metrica':     'Stock de seguridad 99%',
    'valor':       round(float(ss_95 * 1.3), 0),
    'descripcion': 'Cubre demanda el 99% del tiempo',
    'unidad':      'unidades'
}])

# ── 4. Inventario y pedido recomendado ───────────────────────────────────────
inventario_pbi = pd.DataFrame({
    'fecha':                        data['fecha'].dt.strftime('%Y-%m-%d'),
    'inventario_real':              data['inventario'].round(0),
    'ventas_dia':                   data['ventas'].round(2),
    'consumo_estimado':             (data['ventas'] / 6).round(2),
    'alerta_stockout':              (data['inventario'] < ss_95).astype(int),
    'inventario_predicho_manana':   round(float(next_inv_pred), 1),
    'pedido_recomendado':           round(float(pedido_recomendado), 1),
    'mae_modelo_inventario':        round(float(mae_inv), 2),
    'r2_modelo_inventario':         round(float(r2_inv), 4),
})

# ── Guardar CSVs (UTF-8 BOM para Power BI) ───────────────────────────────────
archivos = {
    'KPIs ejecutivos':       (kpis_pbi,           f'reportes/pbi_kpis_{ts}.csv'),
    'Predicciones vs reales':(predicciones_pbi,   f'reportes/pbi_predicciones_{ts}.csv'),
    'Riesgo actuarial':      (riesgo_pbi,          f'reportes/pbi_riesgo_{ts}.csv'),
    'Inventario':            (inventario_pbi,      f'reportes/pbi_inventario_{ts}.csv'),
}

for nombre, (df, ruta) in archivos.items():
    df.to_csv(ruta, index=False, encoding='utf-8-sig')
    print(f"✓ {nombre:<25} → {ruta}")

print("\n✓ Archivos listos para importar en Power BI Desktop")
print("  Ruta: reportes/pbi_*.csv")